In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
    pipeline
)
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel
)

from sklearn.metrics import (
    accuracy_score,
    recall_score,
    precision_score,
    f1_score,
    classification_report, 
    confusion_matrix
)

## Enable if upsampling is used 
# from sklearn.utils import resample 
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from peft import LoraConfig
import pandas as pd
import numpy as np
import os
import torch
import json


## Preparing Training Data

In [20]:
cleaned_sentment_dataset = pd.read_pickle('../data/finnsentiment_cleaned_unambiguous.pkl')

# Replace with your dataset loading
ds = cleaned_sentment_dataset

In [21]:
ds.head(10)

,label,text
2,1,40.
3,2,Kyseessä voi olla loppuelämäsi nainen.
4,2,Sinne vaan ocean clubiin iskemään!
5,2,Itsekin pidän Keskustan kampanjointia ihan hyv...
6,1,"Kamppi, Kontula, Kluuvi"
8,0,en haluaisi että kissani vuotaa.. =)
9,0,Nyt olisi lääkitys paikallaan.
10,1,Ihmiset joutuvat joskus monenlaisien päätelmie...
11,0,"Eniten pelkään sitä, että jos mies vain koko a..."
12,2,Muutenkin suosittelen kaikille asiasta kiinnos...


In [22]:
ds["label"].value_counts()

label
1    12374
2     1358
0     1230
Name: count, dtype: int64

In [23]:
df_0 = ds[ds['label'] == 0]
df_1 = ds[ds['label'] == 1]
df_2 = ds[ds['label'] == 2]

# Undersample class 1
df_1_balanced = df_1.sample(n=min(len(df_0), len(df_2)), random_state=42)

# Combine for balanced dataset
balanced_ds = pd.concat([df_0, df_1_balanced, df_2])

# balanced_ds = ds

In [24]:
# Upsample classes 0 and 2
# df_0_upsampled = resample(df_0, replace=True, n_samples=len(df_1), random_state=42)
# df_2_upsampled = resample(df_2, replace=True, n_samples=len(df_1), random_state=42)

# balanced_ds = pd.concat([df_1, df_0_upsampled, df_2_upsampled])

In [25]:
balanced_ds["label"].value_counts()

label
2    1358
0    1230
1    1230
Name: count, dtype: int64

In [26]:
balanced_ds

,label,text
8,0,en haluaisi että kissani vuotaa.. =)
9,0,Nyt olisi lääkitys paikallaan.
11,0,"Eniten pelkään sitä, että jos mies vain koko a..."
17,0,Tuntuu kuin olisin pelkkä huora.
22,0,Missä kohtaa olen sinua nimitellyt?
...,...,...
23624,2,"Siitä huolimatta naimme salassa, jotta hyvä tu..."
23633,2,Hyvää ystävänpäivää
23637,2,Kiitos kun nostatit hymyn huulilleni jälleen :)
23758,2,Paluuhali Sinulle;)


## Define a Common Config

In [ ]:
# Configuration
class Config:
    # Model settings
    model_name = "TurkuNLP/finnish-modernbert-large"  # Use base for resource efficiency
    
    # # QLoRA settings
    # load_in_4bit = True
    # bnb_4bit_compute_dtype = torch.bfloat16
    # bnb_4bit_use_double_quant = True
    # bnb_4bit_quant_type = "nf4"
    
    # LoRA settings
    lora_r = 16  # Rank - higher = more parameters but better performance
    lora_alpha = 32  # Scaling factor
    lora_dropout = 0.1
    lora_bias = "none"
    target_modules = ["query", "key", "value", "dense"]  # Which layers to apply LoRA
    
    # Training settings
    max_seq_length = 512
    batch_size = 8  # Adjust based on your GPU memory
    gradient_accumulation_steps = 4  # Effective batch size = 8 * 4 = 32
    learning_rate = 2e-4  # Higher than standard fine-tuning due to LoRA
    num_epochs = 3
    warmup_steps = 100
    weight_decay = 0.01
    logging_steps = 10
    save_steps = 500
    eval_steps = 500
    
    # Data settings
    num_labels = 3  # negative, neutral, positive
    label_names = ["negative", "neutral", "positive"]
    
    # Output settings
    output_dir = "../data/models/qlora_modernbert_results"
    logging_dir = "./logs"
    
config = Config()

## Model and Tokenizer Setup

In [ ]:
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=config.load_in_4bit,
#     bnb_4bit_compute_dtype=config.bnb_4bit_compute_dtype,
#     bnb_4bit_use_double_quant=config.bnb_4bit_use_double_quant,
#     bnb_4bit_quant_type=config.bnb_4bit_quant_type,
# )

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(config.model_name)

# Load model with quantization
model = AutoModelForSequenceClassification.from_pretrained(
    config.model_name,
    num_labels=config.num_labels,
    load_in_8bit=False,
    # quantization_config=bnb_config,
    device_map={"": "mps"}, # change to "cuda" or "cpu" depending on system
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
)

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

# Print model info
print(f"Model loaded: {config.model_name}")
print(f"Model parameters: {model.num_parameters():,}")
print(f"Model device: {next(model.parameters()).device}")


Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at TurkuNLP/finnish-modernbert-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded: TurkuNLP/finnish-modernbert-large
Model parameters: 401,208,323
Model device: mps:0


## Tokenize and Split Dataset for Training and Evaluation

In [ ]:
# Split your DataFrame
train_df, temp_df = train_test_split(balanced_ds, test_size=0.3, random_state=42, stratify=balanced_ds['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

# Convert to Hugging Face Dataset
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

# Preprocess using map (not pandas apply)
def preprocess(batch):
    enc = tokenizer(batch["text"], truncation=True, padding="max_length", max_length=8192)
    enc["label"] = batch["label"]
    return enc

train_dataset = train_dataset.map(preprocess, batched=True)
val_dataset = val_dataset.map(preprocess, batched=True)
test_dataset = test_dataset.map(preprocess, batched=True)

# Create DatasetDict for Trainer
ds_preprocessed = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset
})

Map:   0%|          | 0/2672 [00:00<?, ? examples/s]

Map:   0%|          | 0/573 [00:00<?, ? examples/s]

Map:   0%|          | 0/573 [00:00<?, ? examples/s]

## Set up LoRA

In [30]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,  # Sequence classification
    inference_mode=False,
    r=config.lora_r,
    lora_alpha=config.lora_alpha,
    lora_dropout=config.lora_dropout,
    target_modules=config.target_modules,
    bias=config.lora_bias,
)

# Apply LoRA
model = get_peft_model(model, lora_config)

# Print trainable parameters
model.print_trainable_parameters()

trainable params: 35,843 || all params: 401,244,166 || trainable%: 0.0089


## Training and Evaluation Setup

In [ ]:
def compute_metrics(eval_pred):
    """Compute accuracy and other metrics"""
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    accuracy = accuracy_score(labels, predictions)
    recall = recall_score(y_true=labels, y_pred=predictions, average="weighted")
    precision = precision_score(y_true=labels, y_pred=predictions, average="weighted")
    f1 = f1_score(y_true=labels, y_pred=predictions, average="weighted")

    return {
        'accuracy': accuracy,
        'recall': recall,
        'precision': precision,
        'f1': f1
    }

# Training arguments
training_args = TrainingArguments(
    output_dir=config.output_dir,
    num_train_epochs=config.num_epochs,
    per_device_train_batch_size=config.batch_size,
    per_device_eval_batch_size=config.batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    learning_rate=config.learning_rate,
    weight_decay=config.weight_decay,
    warmup_steps=config.warmup_steps,
    logging_dir=config.logging_dir,
    logging_steps=config.logging_steps,
    eval_strategy="steps",
    eval_steps=config.eval_steps,
    save_steps=config.save_steps,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    dataloader_pin_memory=False,  # Can help with memory on some systems
    gradient_checkpointing=True,  # Save memory at cost of speed
    fp16=torch.cuda.is_available(),  # Use mixed precision if available
    report_to=None,  # Disable wandb for now
    remove_unused_columns=True,
)

In [32]:
def predict_sentiment(text, model, tokenizer):
    """Predict sentiment for a single text"""
    model.eval()
    
    # Tokenize
    encoding = tokenizer(
        text,
        truncation=True,
        padding='max_length',
        max_length=config.max_seq_length,
        return_tensors='pt'
    )
    
    # Move to device
    input_ids = encoding['input_ids'].to(model.device)
    attention_mask = encoding['attention_mask'].to(model.device)
    
    # Predict
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=-1)
        predicted_class = torch.argmax(logits, dim=-1).item()
    
    return {
        'predicted_label': config.label_names[predicted_class],
        'predicted_class': predicted_class,
        'probabilities': {
            label: prob.item() 
            for label, prob in zip(config.label_names, probabilities[0])
        },
        'confidence': probabilities[0][predicted_class].item()
    }


## Main Training Pipeline

In [33]:
def main_training_pipeline():
    # """Main training pipeline"""
    
    print("=" * 50)
    print("STARTING QLORA MODERNBERT FINE-TUNING")
    print("=" * 50)
        
    # 5. Create trainer
    print("\n5. Creating trainer...")
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )
    
    
    # 6. Start training
    print("\n6. Starting training...")
    print(f"Training for {config.num_epochs} epochs")
    print(f"Effective batch size: {config.batch_size * config.gradient_accumulation_steps}")

    # Train!
    trainer.train()
    
    # 7. Save final model
    print("\n7. Saving final model...")
    final_model_path = os.path.join(config.output_dir, "final_model")
    trainer.save_model(final_model_path)
    tokenizer.save_pretrained(final_model_path)
    
    # 8. Evaluate on test set
    print("\n8. Evaluating on test set...")
    predictions, label_ids, metrics = trainer.predict(test_dataset)
    print(f"Test Results: {metrics}")
    
    print("\n" + "=" * 50)
    print("TRAINING COMPLETED SUCCESSFULLY!")
    print(f"Model saved to: {final_model_path}")
    print("=" * 50)
    
    return model, tokenizer, metrics

## Define Utility Functions

In [34]:
def save_config():
    """Save configuration to JSON"""
    config_dict = {k: v for k, v in config.__dict__.items() if not k.startswith('_')}
    # Convert non-serializable items
    # config_dict['bnb_4bit_compute_dtype'] = str(config_dict['bnb_4bit_compute_dtype'])
    
    with open(os.path.join(config.output_dir, 'training_config.json'), 'w') as f:
        json.dump(config_dict, f, indent=2)

def load_trained_model(model_path):
#     """Load a trained LoRA model"""
#     # Load base model
#     bnb_config = BitsAndBytesConfig(
#         load_in_4bit=config.load_in_4bit,
#         bnb_4bit_compute_dtype=config.bnb_4bit_compute_dtype,
#         bnb_4bit_use_double_quant=config.bnb_4bit_use_double_quant,
#         bnb_4bit_quant_type=config.bnb_4bit_quant_type,
#     )
    
    base_model = AutoModelForSequenceClassification.from_pretrained(
        config.model_name,
        num_labels=config.num_labels,
        # quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.bfloat16,
        trust_remote_code=True
    )
    
    # Load LoRA weights
    model = PeftModel.from_pretrained(base_model, model_path)
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    
    return model, tokenizer

## Execute the Pipeline

In [35]:
# Create output directory
os.makedirs(config.output_dir, exist_ok=True)

# Save configuration
save_config()

# Run training
model, tokenizer, test_results = main_training_pipeline()

# Test prediction on sample texts
print("\n" + "=" * 50)
print("TESTING PREDICTIONS")
print("=" * 50)

sample_texts = [
    "The company's quarterly earnings exceeded all expectations with strong revenue growth.",
    "Stock prices fell dramatically due to market uncertainty and economic concerns.",
    "The financial report shows stable performance with no significant changes."
]

for text in sample_texts:
    result = predict_sentiment(text, model, tokenizer)
    print(f"\nText: {text}")
    print(f"Predicted: {result['predicted_label']} (confidence: {result['confidence']:.3f})")
    print(f"Probabilities: {result['probabilities']}")

/var/folders/vs/mq6j80s94v7_ts2x082jdx1w0000gn/T/ipykernel_19670/531005792.py:10: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.


STARTING QLORA MODERNBERT FINE-TUNING

5. Creating trainer...

6. Starting training...
Training for 3 epochs
Effective batch size: 32


/Users/veikka/thesis-work/notebooks/.venv/lib/python3.13/site-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Step,Training Loss,Validation Loss



7. Saving final model...

8. Evaluating on test set...


Test Results: {'test_loss': 0.5391779541969299, 'test_accuracy': 0.7835951134380453, 'test_runtime': 35.6422, 'test_samples_per_second': 16.076, 'test_steps_per_second': 2.02}

TRAINING COMPLETED SUCCESSFULLY!
Model saved to: ../data/finbert_sentiment_model/qlora_modernbert_results/final_model

TESTING PREDICTIONS

Text: The company's quarterly earnings exceeded all expectations with strong revenue growth.
Predicted: neutral (confidence: 0.977)
Probabilities: {'negative': 0.0026421311777085066, 'neutral': 0.9771478176116943, 'positive': 0.020209992304444313}

Text: Stock prices fell dramatically due to market uncertainty and economic concerns.
Predicted: neutral (confidence: 0.931)
Probabilities: {'negative': 0.0290075670927763, 'neutral': 0.931078314781189, 'positive': 0.03991406783461571}

Text: The financial report shows stable performance with no significant changes.
Predicted: neutral (confidence: 0.894)
Probabilities: {'negative': 0.017471427097916603, 'neutral': 0.89375036954879

In [37]:
predict_sentiment("nokia julkaisi tänä vuonna positiivisen tulosvaroituksen", model, tokenizer)

{'predicted_label': 'neutral',
 'predicted_class': 1,
 'probabilities': {'negative': 0.017438821494579315,
  'neutral': 0.9181854128837585,
  'positive': 0.06437578052282333},
 'confidence': 0.9181854128837585}

In [38]:
predict_sentiment("nokian tulosraportti ylitti analyytikoiden odotukset", model, tokenizer)

{'predicted_label': 'neutral',
 'predicted_class': 1,
 'probabilities': {'negative': 0.02441665157675743,
  'neutral': 0.956210732460022,
  'positive': 0.01937260851264},
 'confidence': 0.956210732460022}

In [39]:
predict_sentiment("tää firma menee kohta konkurssiin", model, tokenizer)

{'predicted_label': 'negative',
 'predicted_class': 0,
 'probabilities': {'negative': 0.6718098521232605,
  'neutral': 0.23420800268650055,
  'positive': 0.09398221224546432},
 'confidence': 0.6718098521232605}

In [45]:
predict_sentiment("tämä on todella hyvä uutinen", model, tokenizer)

{'predicted_label': 'positive',
 'predicted_class': 2,
 'probabilities': {'negative': 0.12195085734128952,
  'neutral': 0.12386699765920639,
  'positive': 0.7541821599006653},
 'confidence': 0.7541821599006653}